# DuckDB

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/duckdb/duckdb.ipynb)

DuckDB is an embedded analytical SQL engine. Use it for local Parquet exploration, transformation tests and portable analytics where one process is an appropriate boundary.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {'duckdb': 'duckdb>=1.0', 'dbt': 'dbt-core>=1.9,<2', 'dbt.adapters.duckdb': 'dbt-duckdb>=1.9,<2'}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Decisions and trade-offs

Connect to a file to persist a database or omit a path for memory-only work. Select only needed columns and filter early. Parquet scans can use column and row-group metadata to reduce reads, but a small fixture is not a performance benchmark. A notebook connection is not a substitute for a multi-user warehouse service.

[DuckDB Python documentation](https://duckdb.org/docs/stable/clients/python/overview)

## Worked example

Run the small fixture and inspect each assertion. Change one input to see which contract fails.

In [ ]:
import duckdb
import tempfile
from pathlib import Path
with tempfile.TemporaryDirectory() as directory:
    with duckdb.connect() as con:
        con.sql("create table orders as select * from (values (1,10),(1,20),(2,5)) t(customer_id,amount)")
        path=str(Path(directory)/'orders.parquet')
        con.execute('copy orders to ? (format parquet)',[path])
        rows=con.execute('select customer_id,sum(amount) from read_parquet(?) group by customer_id order by customer_id',[path]).fetchall()
        assert rows==[(1,30),(2,5)]
        print(rows)

## Practice

Add a failure case, explain the expected outcome before running it, and identify which guarantee would need a real service or a larger dataset to verify.